# Day 31: Flash Attention — Tiling in Python

**Layer:** Implementation | **Prerequisite:** Day 30 (SDPA)


## Concept Overview

FlashAttention avoids materializing the full O(n²) attention score matrix in HBM by computing attention in tiles that fit in SRAM. Each tile computes a partial softmax using the online softmax trick (tracking running max and sum), then updates the output. The key result: O(n) HBM reads/writes instead of O(n²).


In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import time

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')


## 1. Online Softmax

The key primitive: compute softmax incrementally without seeing all values first.
$$m_i = \max(m_{i-1}, x_i), \quad d_i = d_{i-1} e^{m_{i-1}-m_i} + e^{x_i - m_i}$$


In [ ]:
def online_softmax(x):
    """Numerically stable online softmax."""
    m = float('-inf')  # running max
    d = 0.0            # running sum
    for xi in x:
        m_new = max(m, xi.item())
        d = d * (m - m_new) ** 2 + (xi - m_new).exp().item()  # update with rescaling
        d = d * np.exp(m - m_new) + np.exp(xi.item() - m_new)
        m = m_new
    return torch.tensor([np.exp(xi.item() - m) / d for xi in x])

# Verify correctness
torch.manual_seed(42)
x = torch.randn(10)
online = online_softmax(x)
standard = F.softmax(x, dim=0)
print(f'Online softmax matches standard: {torch.allclose(online, standard, atol=1e-5)}')
print(f'Max difference: {(online - standard).abs().max().item():.2e}')


## 2. Tiled Flash Attention


In [ ]:
def flash_attention_python(Q, K, V, block_size=32):
    """
    Simplified Flash Attention in pure Python/PyTorch.
    Q,K,V: [seq, d_head] (single head, no batch)
    """
    T, d = Q.shape
    scale = d ** -0.5
    O = torch.zeros(T, d)
    l = torch.zeros(T)   # normalization factors
    m = torch.full((T,), float('-inf'))  # running max

    for j_start in range(0, T, block_size):  # key/value block
        j_end = min(j_start + block_size, T)
        Kj = K[j_start:j_end]  # [block, d]
        Vj = V[j_start:j_end]  # [block, d]

        for i_start in range(0, T, block_size):  # query block
            i_end = min(i_start + block_size, T)
            Qi = Q[i_start:i_end]  # [block, d]

            # Compute attention scores for this tile
            Sij = Qi @ Kj.T * scale  # [Qi_block, Kj_block]

            # Causal mask: query position must be >= key position
            qi_range = torch.arange(i_start, i_end).unsqueeze(1)
            kj_range = torch.arange(j_start, j_end).unsqueeze(0)
            causal = qi_range >= kj_range
            Sij = Sij.masked_fill(~causal, float('-inf'))

            # Online softmax update
            mi_new = torch.maximum(m[i_start:i_end], Sij.max(dim=1).values)
            Pij = torch.exp(Sij - mi_new.unsqueeze(1))  # [block, block]
            # Correct previous output for new max
            correction = torch.exp(m[i_start:i_end] - mi_new)
            O[i_start:i_end] = O[i_start:i_end] * correction.unsqueeze(1) + Pij @ Vj
            l[i_start:i_end] = l[i_start:i_end] * correction + Pij.sum(dim=1)
            m[i_start:i_end] = mi_new

    # Normalize
    return O / l.unsqueeze(1)

# Verify against standard SDPA
torch.manual_seed(42)
T, d = 64, 32
Q = torch.randn(T, d); K = torch.randn(T, d); V = torch.randn(T, d)

out_flash = flash_attention_python(Q, K, V, block_size=16)
# Standard reference
scores = Q @ K.T / (d**0.5)
mask = torch.tril(torch.ones(T, T))
scores = scores.masked_fill(mask==0, float('-inf'))
out_std = F.softmax(scores, dim=-1) @ V

print(f'FlashAttention matches standard: {torch.allclose(out_flash, out_std, atol=1e-4)}')
print(f'Max difference: {(out_flash - out_std).abs().max().item():.2e}')


## 3. HBM Access Comparison


In [ ]:
def count_hbm_accesses(T, d, block_size, dtype_bytes=2):
    """Count HBM read/write bytes for standard vs flash attention."""
    # Standard attention
    qkv_read = 3 * T * d * dtype_bytes
    attn_write = T * T * dtype_bytes  # write scores
    attn_read = T * T * dtype_bytes   # read for AV
    out_write = T * d * dtype_bytes
    standard = qkv_read + attn_write + attn_read + out_write

    # Flash attention (tiled)
    n_q_blocks = (T + block_size - 1) // block_size
    n_k_blocks = (T + block_size - 1) // block_size
    # Each Q block reads all K,V blocks; O is accumulated in SRAM
    flash = qkv_read + n_q_blocks * (n_k_blocks * 2 * block_size * d * dtype_bytes) + out_write
    # Simplified: dominant term is QKV read + output write (no attn matrix in HBM)
    flash_simplified = qkv_read + out_write  # ideal
    return standard / 1e6, flash_simplified / 1e6

print(f'{"Seq":>8} {"Standard (MB)":>16} {"Flash (MB)":>14} {"Reduction":>12}')
for T in [256, 512, 1024, 2048, 4096, 8192]:
    std, flash = count_hbm_accesses(T, d=64, block_size=32)
    print(f'{T:>8} {std:>16.2f} {flash:>14.2f} {std/flash:>11.1f}x')


## Experiments

1. Measure tile size sensitivity: sweep block_size from 16 to 256. When does a larger block hurt (doesn't fit in SRAM)?
2. Implement backward pass tiles. What additional state needs to be stored for the backward pass?
3. Compare against triton.ops.flash_attention for the same inputs.


## Key Takeaways

- FlashAttention tiles Q,K,V into SRAM-sized blocks, computing attention without ever materializing the full n×n matrix in HBM.
- The online softmax trick makes tiled computation exact: running max and sum are maintained and used to correct previous tile outputs.
- HBM access is reduced from O(n²) to O(n) — the dominant speedup source for long contexts.
- The real Triton/CUDA FlashAttention goes further: double-buffering, async memory transfers, and warp-level specialization on H100.

## References
- Dao et al. (2022), "FlashAttention"
- Dao et al. (2023), "FlashAttention-2"
- Milakov & Gimelshein (2018), "Online normalizer calculation for softmax"
